# 📋 STR01 · 미션 10 | 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성

**학생용 Student · 직접 실습**  
CMS 편성: 4시간 · 교육용 가상 데이터 사용

---

## 📖 1. 상황과 이번 미션의 문제

가상의 20개 조직과 5종 물품의 기록을 이용해 **전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성** 과제를 수행합니다.

실제 조직의 기록이 아닌 실습자료이며, 코드를 실행한 결과도 실제 현장 상태를 뜻하지 않습니다.

**이번 미션에서 해결할 문제:** 환경 점검 후 데이터를 검증하고 피처를 만들어 예측·경보를 산출한다.

경계 이상 건과 이미 재고가 0인 건을 모두 군수처장 승인 대기로 분리하며 자동 집행하지 않는다.

주기마다 도전 모델을 학습해 보류 구간 성능이 2% 이상 좋아질 때만 교체하는 재학습 파이프라인을 4회 실행하고, 운영 가이드 9장 목차와 자체 점검표, 브리핑 요약을 출력한다.

**만들 결과물:** `docs/str01_operations_guide.md`, `pipeline/str01_retrain.py`, `outputs/M10/reg.csv`, `reports/str01_briefing.md`, `forms/STR01_M10_forms.md`

## 📌 2. 처음 시작하는 방법

1. 아래 **환경 준비 셀**을 실행합니다. Colab에서는 메뉴의 **런타임 → 모두 실행** 대신 각 셀의 결과를 먼저 확인하면서 진행해도 됩니다.
2. 핵심 함수의 `TODO`를 AI와 함께 구현합니다. 처음에는 함수의 입력·출력과 왜 필요한지 질문하세요.
3. 첫 실행 결과를 확인하고 실패하면 오류 메시지를 AI에게 설명해 달라고 요청합니다.
4. AI가 제안한 코드 중 최소 3가지를 원자료와 실행 결과로 대조하고 수정 이유를 기록합니다.
5. 활동별 작성 칸을 채운 뒤 **실제 AI 대화 원문**을 보존하고 마지막 저장 셀로 ZIP을 만듭니다.

> **보안과 평가:** 실제 조직·위치·재고·작전 자료, 개인정보를 외부 AI에 입력하지 않습니다.

- 모델 성능은 **실제 출력 로그**로 판단하며, 예시값을 실측으로 적지 않습니다.
- 실행 성공이나 ZIP 저장은 학습 PASS·실제 승인과 다릅니다.
- 폐쇄망에서는 허용된 Python/Jupyter 환경에서 실행하세요.
- 인터넷이 필요한 `pip` 설치는 폐쇄망에서 실패할 수 있으므로 사전 패키지 준비가 필요합니다.


In [ ]:
# @title ① 실습자료 자동 준비 및 환경 설정 (ZIP 업로드 불필요) (실행 버튼을 누르세요) { display-mode: "form" }
# 이 셀은 코드 숨김 상태로 배포됩니다. 소스코드를 열지 않아도 실행할 수 있습니다.
from pathlib import Path
import os, sys, json, re, math, time, warnings, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 업로드 파일명은 자유이며, .zip 확장자와 유효한 ZIP 내용만 확인합니다.
current_dir = Path.cwd().resolve()
notebook_directory = globals().get('notebook_directory', current_dir)
workspace_name = 'STR01_실습프로젝트_작업공간'
WORKSPACE = next(
    (folder for folder in (*reversed(current_dir.parents), current_dir) if folder.name == workspace_name),
    current_dir / workspace_name,
)
STR01_PROJECT_READY = False

def _prepare_project_zip(raw_bytes):
    global STR01_PROJECT_READY, STR01_PROJECT_DIR
    import io
    if not zipfile.is_zipfile(io.BytesIO(raw_bytes)):
        raise ValueError('유효한 ZIP 파일이 아닙니다. 압축파일 내용을 확인하세요.')
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as archive:
        for member in archive.infolist():
            name = member.filename.replace('\\', '/')
            target = (WORKSPACE / name).resolve()
            if not target.is_relative_to(WORKSPACE):
                raise ValueError('안전하지 않은 압축 경로가 발견되어 중단했습니다: ' + name)
            # 심볼릭 링크는 작업공간 외부 접근 가능성을 고려해 거부합니다.
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('심볼릭 링크가 포함된 ZIP은 허용하지 않습니다: ' + name)
        for member in archive.infolist():
            target = (WORKSPACE / member.filename.replace('\\', '/')).resolve()
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if target.is_file():
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('xb') as destination:
                import shutil
                shutil.copyfileobj(source, destination)
    os.chdir(WORKSPACE)
    STR01_PROJECT_DIR = WORKSPACE
    STR01_PROJECT_READY = True
    print('✅ 실습자료 준비 완료 (기존 파일은 보존):', WORKSPACE)
    print('✅ 이제 다음 실습 코드셀을 실행할 수 있습니다.')

# 실습자료가 노트북 안에 내장되어 있어 ZIP 업로드 없이 자동으로 준비됩니다.
# 내장 파일 2개: STR01_M10_forms.md, STR01_M10_lesson.md
import base64 as _embedded_b64
_EMBEDDED_ZIP_B64 = (
    'UEsDBBQAAAAIAAAAIVyPDLIARAkAANgVAAASAAAAU1RSMDFfTTEwX2Zvcm1zLm1k5VhbT1vZFX73r9hSFKmVDDZMKs0gVVXUixRpMpWavo0qRAZPigQY2WSq'
    'SjyYcIgc7FHM1AdscozsCQRMPcrBHMBIppX4KZm3s/f5D/3W2nvbx2aS9GH6VHHx8b6uy7e+tda5Ix79+U/pKfFwKi3U/rZyToXa2ValhnhXqArVdFSxKeRF'
    'QZYLQp4FYe+FUMWaOij84nd/fPRLoUoHquSFviPkybEsFSPHl0frQvovRdj1I/dbfFxjHxZ6ausg2vRE9Pw8cttC1R1clkioVxV51hPRyxehX5gRX0Yurqz9'
    'xS6LH3tzqfYr8mUdgnYit662zoWWcnw0KpdVI4iqjmz0VaOHZ2jiqo11kjfaeSErtfCsKdS5p77fFLLTUw50KIipdPouCSavHCxRu0Xl9WXbTwq1F6jaOp9z'
    'VFDPoZVfkFuHkcvnhOc+RKYroKnqBuGZo+oFoXZ9iEVrer5QLzo4BEs7quVDJhwwmUjIswLsI8Kzvmp6M5itYDsJFDmnquGI6btqtwLhRPqubHkwazX8dxWX'
    '11WjIO5Fe2VSWm00jF6qASmcZrTRwIlC1Q6UfywP+wL64GiYE6aUpYNJo4/RXHwW+h4OOqTt8AWN4m6NBqtit6De6GvKx/BDI5Bve8bF+Lp1BZUEMEKqksdb'
    'Ht2rmhVsHN6bSNy5E3MTK4AdstVIJNboCbhSjb5YE3rFzSUMKg+1aUvwa93Fk5DdqvIcWlVzoHVKvnQAVPHw/u8xpm2YUl4Trrq5DC/74amPcXVdlf+ACbu8'
    'dg33TUxMiPf+x/xU+h6+xX5pbGrqJ8Y+vT02/auxMVZdA85YX1uHNI/cH+RJm4REODXhizcNIzSw+Lqcgq1jT70WGQG7XjkERcLTVe1DGmFG7W/K5qFxIxzG'
    'fhTqYFuWjsckT6fDU2DwwiHDj0yNeN0G9uiKOBCG4Xq0DTelR5d+BsCN4VBjb3TZ/Qd0WlSv4jTwTtOlUzsGjiMrf5tdnHtsAmFs5uEjobrHYDFsbhbo4pF5'
    'QCYMCmSc6Hlz3GcQALpEe4A5wkT6Hj717Ra5E0I96xCoonKFzGlpFIzXrMyIdzst1qJ0jmjXHOoTsfKEGVW7gdwvk0fNbhx6/wFiX8ijVtjrYcGm7ATMKo0A'
    '2J6hW5teeAaSbDIBvi5DOcY4Kekp95o54BXCvia3quQ1DNA+mNmSYq0IUoxcz8zy+eBfYnDw7mFAG6Ec8DlDONWewFxbNddjcN2rqu4PdBe5T9YqvE0bGoHK'
    '6LRrrcIfXGslBB21q5DkI+Ae9eaIA7Vo4BG+lljTyswCEM8RZ7eCODkaptDcbES0Pmc7Maf6NUCXbEcH1B1jBXCRAUac9hjHyjmQJ5sDnfUtmq7JPtaHDOCb'
    'y2izTAlzC/FmKWxoFo00djDDgfYbwrNRYphzbZDlWF6dGvtYZmz9X/DgLXL72EDiE2IQkmxjnS1cLoCk4hZmXcHFxajW40Th/0sbeYMwH172IBu0AforlH0i'
    'N5ClKyGfBWqvTSz0rGb9SLn2mW/NFvMPEIoMEvOA9ieDHf7ZOoSJy2UUCKA0lBfCLuDQtIFks4Wd4wAdn8N/Ho9LEMO94XztZ+uun8HsnFB08aQzJ19rbgMB'
    'XHicWriuGmYYs9T/DgqA4k+QzDFEBdwgRLUGsdQpSx1QEja87eGR7FbhNHxRpKPXbAD8T7B0W+0JVjIoDhxhvlKEBUXlFk1EGxcwU1qXbfYQUPJincMDyELW'
    'IT178g30RdGCR2O72G4ASDWAVqTezRpVhFSJOTWiZejPwTYk19FcQuy64ekcYKojfowzBo/oxPljoSpLBHUECBVZfeQQ64cfCy67PHLPValDhSSqn1j1FLnH'
    'VITaVI/tFx6lfKqZGpRUVfefwpKLJgkRg+2Qe30v7HYoTFA/gkEGiy0cPkjF+b9mny7Oz67kskvZ1Qwm+AeF7tuekViXHsyAzinCcIhIzkiuI0v1AU+QyM4p'
    'spwRkyF+USXYxSPSFL/OrUJXW6zuoQqRZ1h45LD6Ywl8zSyhmkpbS8sUCwnWfciaxrEMo49An0oqPFN/Ba5CHTEj5Lc+tSeOb7IKChP6Qtm410qawio5Wkxx'
    'Qfamk2TIlTyk4fCqTBRJjLIT4IlMip5srGN6TxfGhvxo2+SYLg1oIGsSN1N7tldleBicafqgFHsM88BGKIb22qZDMlUQXTcKDWQxHGAwSiTuXvN9nCNN0zHI'
    'd7q+0knv5hKFEdyw71g/arPSbpN4d4qwBe7/jUHfmpi2HmAMzoiVhZXM4sJyJpVfzaWnZnOZ1dzcwvLkyt8puY4KilyiyxMtlBYVgbo0l5nNLs4n6XM587fk'
    '0sLy7MISNn2TWcosr/56Mj0t3j3/zvZ0arcDN4vH2eziJOVqWBC/aZ0kY60aqUQFFxsA4Cae8qlAdMJuO9ovcn1IXWlAFTP2Ah7Td02yhdE9SjRuoNy+udn4'
    'TlAIHDZB3twzpbkU4pLB3PXF3BcpeVU0CYNFOfUJmXo/6AcWVhdt1To1FSWiUwenZRVNoLhi20tpUsUWU14NfUIQe+V8nsnns8vkWw5nHc10AwecJmzgzVRZ'
    'tz36ifUoQ+D/672BYaVB4Jk8jgWouzRfX9XgDJ2nzWU6S6Y0mQ8Y0xBmzDvWGAQQdMApdLz4+zSFjpYSEuIDwUyJQL95IBYwbG3a8OH2QUM/MXXPshU38mQQ'
    '3cvLiicAF0R6UpiK1QBQ19F6/8jmWzvlUVmo6roV43Y0UElzrfb8QURo6+u3KGAn2ATxNAiHYZk6SsAnbZBLvIXgadib21vTq3OwjiQx29yiafKP+V0LBDZl'
    'xHAP4YZUdLeGVSuT5/62PeEP9x98zhxpwxYQG7y7sc2zaT4pLWrMAB+mbHZMVRLXUL8H0IHLMW9al4Dsr0Nq9N0PyWQyFZuC3yDETqQq7HUfAqV0xUutAK7W'
    'CZTuYRFs4uRmXoPxJ2L8nolxbaUZMZ/9Km8YO7uSyc2tLmSX87NPni7MZyaX5pPvZ/WkyD5dXXm6mk/huFQu82Tyq/w3SZHLrGRzq/bIx7mFzNcLy0/4qK+z'
    'uaV8il+NzmLPLH/HDBGWZii2sy4+v9/U9TR3x5rvdKGAaTaCrvopc+/6SMjwtVXVltSD9zp4ZG/TgL1CX8ho01jknOQiDpAQEANUD4xb7z9QSwMEFAAAAAgA'
    'AAAhXJV+9kqrCgAAzBsAABMAAABTVFIwMV9NMTBfbGVzc29uLm1k7VnRbhvXEX3nV1zAMNACNCnKTpEIKAqjSVujcRokfWpRSHLEuAQkUZDkBH2jrJVBi1RN'
    'OVyLlJcqFVOW5LDoiiIlCqBRwJ8Sv/He/Yeembm75Fa2A/StSAAbEpe7986dc87MmdUV9fkfP5vIqNuZCfV9oapM0zHFptJnBV0uKH3aHfYfKlOsmVbhZx/+'
    '4fOfK1NqmZI39B2lXxzpUjFwfH24prT/SA07fuBu4cdLPIcbPbPZCjY8FTzoBe6xMnXHOCeJxJUryqyvBfXj4ekAixwH214iMfQLuKh+/+kt7I37GkqXurp0'
    'YfYqet8zjmdD0qeOMnvt4Wnz1bl5UEYMr85lDYXwTL0Qj0IfDOx6iADR6lIrpfSWbxpdBI6bCuZ5AyvTB1Mrmv5+0j6ZVGbzwjT6CpsO+z6top+38S0CLXlB'
    'vTq8KJvDgsJK5kkXv+nNKoUQS8ur87fkhI5OR3tUp9MEbt1s9lRQLlNcVUc3BtjZNBybs8D1KHBJ3dOKPu0TJMGTh0p3qsZz6NjBo4cILZG4Ft4RuECypvS/'
    '+majlmJgO12Gb7NndipIKUUs64ySEz2P9K4X1W9zqyr4ex8xqOup9wT414UqrdLfH/qPEa8yrqNLtdcFd7S3rDqlgnoNJ8feFWRaBU+dUe4JGUl/4NYApwqq'
    'A9OpMWCHZf1NC1m1kFAWQzDNfd+ceTYhlNYhEQIYgDzDky7ntdHFoS1LkBM1QTDiS0ooobNdUcOeT6Tu1MzeQRxosM1ijbj0sR/BdLjNiaox6O4mkschmGcD'
    'euqQsqf0Iwd55k22GD9BNnC7iuOvYeM2aQcJ1ZsHlL3Jq2H45tsSUmkOi0q7ZeQA91YAGYVBQL2LJjeC3TJBKxHi2GDvbtfU1piuyMc3nvqAjgqpGP+I+IJj'
    'MR8YGsgnqXS/zKfexrNV475k/p15unkQ8u+a+rOw6i+hpuNkv8RoKSk/yHMKw+UU8BF0hQlhep75dkPpNujnEY6ZiYmrpAh94eAWs1M03gAIhWfldQ4L5sEW'
    'nRrptcQSrGkLYIAzgzGcgB0fYdE9ULd52MYihI7ZB0H7WCA1RmcmnW4PptRc/ouV9Mrq8kRmOr+UXZ5dzeUXV6bv3svNZVMLc1wGl3JL2fncYtbetpxdXZ7N'
    'LaaW/ib6bUL8PU7KszLJFZ+QTFzYJpUZt6jvd1+d/zo/P3vH5iO6KOJVeqMfbPb12Vp4hVKJItls4LL6061POfSXVaA+pWbeGs5MCvTsmVIbP46QIVZkrx3U'
    'HKzbLIAuNoGm810YCUn45i1O31mVMNGnBYgwVj8iiUP/44HNzM2uzqY/++jmh7c/Qq5mWOpHA2wFBbzEMewSSDr2/mDoe1TDcAhU+KprNqraKYKgmUzsG5Ya'
    'srcBnmZ+ga8QvISGWAH4+7hkTwZZxwsdqvEe2P5k25QayBR3wml0wukv88sLKxQjA8qYL2eX8surK6YhJZXrcghaUAJbBF1orVO0RZfutRmWvBLcTP7BO3Gh'
    'uxgR3LXy1/y9+bnppeX8Qn41O8N8cSDOKbUwm53Oz88l6edi9uvkQm5xOreAG7/KLmQXV3+ZmphU3z94bKsI6I7zF9SdfH4eRctrQtP4NxHWnloLdSGsyffb'
    'YU3mFFIyzY4z7BwHe0U+M+muS00Zz6JsjmoYkm0crN017iCqX1KrITZ9gK6NPIBJ2LhAYUCwdq9PZj9J64uibWQcyolPrTkqPtqvaUhn1L7NeoPTL/SlImnF'
    'xdwnTtiOH15fX8ONCIHJw0WnwbQ3fh+6pzJ9O0OofTU7nwNbs9Pz2bm72eWRMmL8iWHL0mClOCe8FwuGWw6qzn43LEZ9Hx/i6gGkUAGzNGzHVBEd7pcXDtI4'
    'Zl5I8U0PIqR+BRsCCZz2ucGdD5Awwi94QOao1qJ+Q12u6TI8XpMVd/9oeLEWHoI4G1blugeI6MoV+3uGyzf4FuxucWO936buFpQrunSUSNjiKO4LPRpdkGol'
    '5156wbMyn6nxkk5ulWrpKOdS5PBKjIQu9dTv5HZp325ZcdneKxKy2AKrU1c0WwNcQPZ+RXdLMGP5QXXawbqH+8N+39qjMBgqax60AwLUXe74/rHuuNznmu4Y'
    'tGM5mOQc2JLHR7OgwbICylZ4lSMfa8BkQ5EzzrjFH08+btml8CySSjsLeyhm01wzu8fjPnWcHVNR5tr9wDmhRyevio9TE1fZtfjV4b+Bl18nbYkl2GtDIxEX'
    'kU2Ic73BYEaKv0wxsQ7SeBUXYhRkrnyAoC3GlCtfdFbycbxNGX7cIijWFx9jPpqs3L6kS6zHG9N+XagnlMaT1JXZtLTlLGCfJfYo+7GWFd44nk72PgpY4Ftu'
    'ndxbWa5QkZwWfRYgWJcRxU4fhcy2jTwpWg6SEI3T0i82cLPwQ3J8HYbTVkXJl2Ug78qTD5/KxkscqUPtpR5Ah4vEbzRmKSHqm3N0Q9yVuCTRgajUmpVIoVJR'
    'lfCV6kVBTDkXmEivb6grsEpNmhM4ux777hFJINl/YAAptZHBJHe8/j4uJkPeCEMoIrBr2C2ENptmv/VYS+7GD8E18w1VCg2nQaOAcO0ZrCMYUMEpQ2NJtM9M'
    '3EhnMhn8fz+dmXzPkMOskYSISCKHsb5g/frocckRnrqWuREOeFybyFwSGPhQ8Ww1Soam3rYswBY9H3v40pNcx6rh+dTl/umRTsyuP5oBWGcibZya+t1xN2qg'
    'I1TiM+uLY73f4EcEXfkaYPMwY2nN7Z1muHqUFqsFEQwVAARMTR9VZvQMeXCZhUbeLfRUdoXf3Lz1MbM/bPQ0UIUFxVaTyEQ9daz/BvVYhVweuZ3F+x+KjEIo'
    'w/M+s4hmNgyRXcq/HXpjBUlxh+FJklPR2o51jNCVg94sHz6fBZIM28y7LT9Zsnd7uZn8vdWle6sraThL2Mi7qS9WvuLr1lLah+4s57Jf5hbv2jXZgtrcMNdG'
    'AqXxrPNPqrM4SS/J6ukW8Xm8JCZDyy6ZgnCQrucD6oFjxmJ8yBFBy8HpNYt0MCq7NA/TixZ8h+EVPzFUktLJQKEVNdescm3BpkVF89xyJG4Bckz92AKIEaVC'
    'BCOMSQWxqiu8tQyEhOgNjHuZfYmEdeFcTtgX1WyxpAPZrh9Cm0nxWy/xN1P/t2+FyDoQ7XerVDZkEBpBRfWUHRC8G/qaDMa8I7ovtovPF1RS9zZk6LeDBu/H'
    'DS5qVX0OoyZ2UJB/dQ5f4zlmzwkrtKSVJSzwRi0zMWkTz85n6u3zMqEXjw9cGG/zEuHrQvVHNgu9LpBpNWfHZv8kfD3U+U6cjKgewFvZQy1pkT4esVZ05F6I'
    'WU+dj7MrK/lFgpTHEjEItANL105TOxXMX/CFY0Bet0Ay4FM/qpdDtqxFMrMVEjfA90kpvqgBA8iDZ4so6G4xLXU6GhCt/x4D5SdD85Oh+d8NDVsE2OO9g/Ro'
    'FMHW0pxpH/uKQIYOUmY4LI2kfUOkLcn5odeeybfX8KR6g/tJqrdan6QYn/Tld3FUnqQecXoRNpeAof8YJ+GRRaobtaGLGr7ms8uLD2rPOz66LiAOVYaJBcML'
    'ng0nl3AGpAvhFrIhk8z+eYA6kAv6o/xbIzJKWmLsTyfv/MtJWpaUP6BEfz9J2bdDMvDjvPKCSbgjF6nAy4uVzep/ayXxH1BLAQIUAxQAAAAIAAAAIVyPDLIA'
    'RAkAANgVAAASAAAAAAAAAAAAAACkAQAAAABTVFIwMV9NMTBfZm9ybXMubWRQSwECFAMUAAAACAAAACFclX72SqsKAADMGwAAEwAAAAAAAAAAAAAApAF0CQAA'
    'U1RSMDFfTTEwX2xlc3Nvbi5tZFBLBQYAAAAAAgACAIEAAABQFAAAAAA='
)
_EMBEDDED_ZIP_BYTES = _embedded_b64.b64decode(_EMBEDDED_ZIP_B64)
print('📦 노트북에 내장된 실습자료를 준비합니다. ZIP 업로드는 필요하지 않습니다.')
_prepare_project_zip(_EMBEDDED_ZIP_BYTES)


## 미션 핵심 개념과 원본 실습 설명

---

## 🎯 미션 10. 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성

**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침

#### 미션 배경 및 당위성

- 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템을 완성한다.
- 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **데이터 검증부터 예측, 경보, 승인 대기 분리, 가시화까지 이어지는 DOS 모니터링·결핍 경보 시스템과 자동 재학습 파이프라인을 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.
- 이 문제를 해결하기 위해 본 미션에서는 **환경 점검 후 데이터를 검증하고 피처를 만들어 예측·경보를 산출 수행하고 검증하는** 실습을 체계적으로 수행합니다.
- 학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/str01_operations_guide.md 및 pipeline/str01_retrain.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)

#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지속 가능한 운영 가능성 및 국방 군수 정보체계와의 연동 용이성

#### 미션 PASS 조건

- 정량 기준: 자동 재학습 파이프라인 정상 실행, 최종 문서 100% 완비

- 진행 기준: 최종 인도물 검수 및 프로젝트 완료 발표자료 완성

- 검증 방법: End-to-End 전체 파이프라인 자동화 실행 및 최종 검수

#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.


### AI 활용 예시 · 미션 10 전용

> 나는 Python을 처음 배우는 학습자야. **전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성**을 진행하고 있어. 다음 요구사항을 보고 핵심 함수의 입력·출력, 필요한 검증 3개를 쉬운 말로 설명해줘. `TODO` 구간만 구현할 수 있는 코드를 제안하고, 데이터를 미리 보지 않는 검증 방법과 경계 사례도 알려줘. 실행 결과는 만들지 말고 내가 실제 실행한 로그를 받은 다음 분석해줘.  
> 요구사항: pipeline/str01_retrain.py / should_promote: mae_old,mae_new,min_improvement=.02 → 교체 여부 bool. 유한한0이상 오차를 검사하고 후보가 엄격히 작으면서 상대2% 이상 개선해야 교체한다. 동률·현행0은 유지하고 NaN/빈 평가를 거부한다.

**AI 답변을 받으면:** 그대로 복사하지 말고 입력 시점·자료 범위·실제 실행 여부를 직접 확인하세요. 확인되지 않은 수치는 `미측정`, 수행하지 않은 활동은 `미실시`로 남깁니다.


In [ ]:
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    """데이터 검증부터 피처 생성, 결핍 위험 예측, 3단계 경보, 군수처장 승인 대기 분리, DOS 현황 가시화까지 한 번에 실행하는 모니터링 도구를 완성하고, 주기적으로 모델을 다시 학습해 성능이 좋아질 때만 교체하는 자동 재학습 파이프라인과 운영 가이드 목차·자체 점검·브리핑을 출력한다."""
    import importlib.util
    import sys
    import time
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.ensemble import GradientBoostingRegressor

    # ---------------------------------------------------------------------------
    # [교육용가상자료] 가상 부대 20개 x 핵심 보급품 5종 일일 재고·소모 시뮬레이터 (실제 부대·수량·기준일수 아님)
    # 핵심성과지표(KPI) = 기준 보급일수(DOS) = 현재 재고 / 최근 평균 일일 소모량. 목표 DOS는 교육용 가정값이다.
    # 결핍 위험 일수 H = 보급이 없다고 가정할 때 현재 재고가 바닥나기까지 남은 일수(미래 실제 소모량으로 계산한 정답).
    # ---------------------------------------------------------------------------
    ITEMS = {"식량": (100, 0.3, 10), "피복": (20, 0.1, 15), "유류": (300, 1.2, 7), "탄약": (60, 1.5, 6), "수리부속": (15, 0.5, 12)}   # (기본 일소모, 템포 민감도, 목표 DOS)
    N_UNIT, DAYS = 20, 160
    UNITS = [f"가상{u:02d}부대" for u in range(1, N_UNIT + 1)]


    def make_data(seed=1, n_unit=N_UNIT, days=DAYS):
        """일 단위 재고 원장: unit, item, day, tempo(실제 작전 템포), plan(사전 계획 템포), wx(악천후), consume, resupply, stock."""
        rng = np.random.default_rng(seed)
        rows = []
        for u in range(1, n_unit + 1):
            s, ph = rng.uniform(0.7, 1.5), rng.uniform(0, 6.28)
            surge = np.zeros(days)
            for st in rng.choice(days - 10, 4, replace=False):
                surge[st:st + int(rng.integers(5, 10))] = rng.uniform(0.4, 0.8)
            tempo = 1 + 0.25 * np.sin(2 * np.pi * np.arange(days) / 30 + ph) + surge
            plan = tempo + rng.normal(0, 0.08, days)
            wx = (rng.random(days) < 0.15).astype(int)
            for it, (b, sens, tg) in ITEMS.items():
                stock, off = tg * b * s * rng.uniform(0.8, 1.2), int(rng.integers(0, 4))
                for t in range(days):
                    c = b * s * (1 + sens * (tempo[t] - 1)) * (1 + 0.12 * wx[t]) * rng.normal(1, 0.08)
                    r = max(tg * b * s - stock, 0) * rng.uniform(0.9, 1.1) * (0.5 if rng.random() < 0.08 else 1) if (t % 4 == off and t > 0) else 0.0   # 4일 주기 정기 보급: 목표 재고까지 채움(가끔 수송 차질로 절반)
                    stock = max(stock + r - c, 0.0)
                    rows.append((f"가상{u:02d}부대", it, t, tempo[t], plan[t], wx[t], c, r, stock))
        return pd.DataFrame(rows, columns=["unit", "item", "day", "tempo", "plan", "wx", "consume", "resupply", "stock"])


    def make_samples(d, lo=21, hi=125):
        """(부대, 품목, 기준일 t0)별 특성(피처)과 정답 H. 피처는 t0 시점까지 알 수 있는 값과 사전 계획·기상 예보만 사용한다."""
        out = []
        for (u, it), g in d.groupby(["unit", "item"]):
            c, stk, rs = g["consume"].values, g["stock"].values, g["resupply"].values
            pl, wx, tp = g["plan"].values, g["wx"].values, g["tempo"].values
            cs = np.cumsum(c)
            tg = ITEMS[it][2] * ITEMS[it][0]
            for t0 in range(lo, hi):
                idx = np.searchsorted(cs, cs[t0] + stk[t0])
                if idx >= len(cs) or idx <= t0:
                    continue
                c3, c7, c14 = c[t0 - 2:t0 + 1].mean(), c[t0 - 6:t0 + 1].mean(), c[t0 - 13:t0 + 1].mean()
                last = [k for k in range(t0, t0 - 15, -1) if rs[k] > 0]
                out.append({"unit": u, "item": it, "t0": t0, "H": idx - t0, "stock": stk[t0], "c3": c3, "c7": c7, "c14": c14,
                            "dos7": stk[t0] / c7, "dos3": stk[t0] / c3, "trend": c7 / c14, "burst": c3 / c7,
                            "plan7": pl[t0 + 1:t0 + 8].mean(), "plan_ratio": pl[t0 + 1:t0 + 8].mean() / tp[t0], "tempo0": tp[t0],
                            "wx_fc": wx[t0 + 1:t0 + 8].mean(), "since_rs": (t0 - last[0]) if last else 15,
                            "rs_avg": rs[t0 - 13:t0 + 1].sum() / 14, "fill": stk[t0] / tg, "item_code": list(ITEMS).index(it)})
        return pd.DataFrame(out)


    def mape(a, b):
        # [TODO : 학생 실습 10] 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    LOG = []
    def stage(name):
        # [TODO : 학생 실습 10] 전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성 관련 핵심 로직을 구현하세요.
        # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
        # <<< 여기에 코드를 작성하세요 >>>
        pass  # 작성 후 pass를 제거하세요.
    mods = ["numpy", "pandas", "sklearn", "matplotlib"]
    env_ok = sum(importlib.util.find_spec(m) is not None for m in mods)
    print(f"[0] 환경 점검: Python {sys.version_info.major}.{sys.version_info.minor}, 필수 모듈 {env_ok}/{len(mods)} 확인 [교육용가상자료]")

    @stage("데이터 검증")
    def validate(d):
        rules = {"중복 없음": not d.duplicated(["unit", "item", "day"]).any(), "결측 없음": not d.isna().any().any(), "소모·재고 0 이상": bool((d["consume"] >= 0).all() and (d["stock"] >= 0).all()),
                 "품목 5종": set(d["item"]) == set(ITEMS), "일자 연속": d.groupby(["unit", "item"])["day"].count().eq(DAYS).all()}
        return rules

    @stage("피처 생성")
    def features(d, lo, hi):
        return make_samples(d, lo=lo, hi=hi)

    FEAT = ["stock", "c3", "c7", "c14", "dos7", "dos3", "trend", "burst", "plan7", "plan_ratio", "tempo0", "wx_fc", "since_rs", "rs_avg", "fill", "item_code"]
    def new_model():
        return GradientBoostingRegressor(n_estimators=80, max_depth=3, learning_rate=0.12, random_state=0)

    def level(h):
        return "심각" if h <= 2.5 else "경계" if h <= 4.5 else "주의" if h <= 7.5 else "정상"

    d = make_data()
    rules = validate(d)
    S = features(d, 21, 125)
    tr = S[S["t0"] <= 90]
    champion = new_model().fit(tr[FEAT], tr["H"])
    now = S[S["t0"] == 100].copy()
    now["pred"] = np.clip(champion.predict(now[FEAT]), 1, None)
    now["lv"] = now["pred"].map(level)

    @stage("HITL 대기 분리")
    def queue(now):
        q = now[now["lv"].isin(["경계", "심각"])].sort_values("pred")[["unit", "item", "pred", "lv"]].copy()
        q["상태"] = "군수처장 승인 대기"           # 자동 집행 없음
        return q
    q = queue(now)
    dep = d[(d["day"] == 100) & (d["stock"] <= 0)][["unit", "item"]].assign(pred=0.0, lv="심각(이미 결핍)", 상태="군수처장 승인 대기")
    q = pd.concat([dep, q], ignore_index=True)          # 이미 재고 0인 건도 승인 대기에 포함

    # ---------------- 결과 출력 ----------------
    print("[1] 데이터 검증:", {k: ("통과" if v else "실패") for k, v in rules.items()} if rules else "실패")
    cnt = now["lv"].value_counts().reindex(["심각", "경계", "주의", "정상"], fill_value=0).to_dict()
    depleted = int((d[d["day"] == 100]["stock"] <= 0).sum())          # 이미 재고가 0인 건은 예측 대신 즉시 '심각(이미 결핍)'으로 분류
    print(f"[2] 100일차 경보 분포(예측 {len(now)}건 + 이미 재고 0 {depleted}건 = 100건): {cnt}, 이미 결핍 {depleted}건")
    dos = now.assign(target=now["item"].map(lambda i: ITEMS[i][2]))
    dos["rate"] = dos["dos7"] / dos["target"]
    print("[3] 품목별 평균 DOS 달성률:", {k: f"{v:.0%}" for k, v in dos.groupby("item")["rate"].mean().items()})
    print(f"[4] 군수처장 승인 대기 {len(q)}건(자동 집행 0건), 가장 급한 3건:")
    print(q.head(3).round(1).to_string(index=False))

    # ---------------- 자동 재학습 파이프라인 ----------------
    @stage("재학습 파이프라인")
    def retrain_pipeline(cutoffs):
        """주기마다: 최근 자료까지로 도전 모델을 학습 -> 직전 14일 보류 구간에서 현행 모델과 MAE 비교 -> 2% 이상 좋아질 때만 교체, 모든 실행을 기록."""
        global champion
        reg = []
        for c in cutoffs:
            trn = S[S["t0"] <= c - 14]; hold = S[(S["t0"] > c - 14) & (S["t0"] <= c)]
            ch = new_model().fit(trn[FEAT], trn["H"])
            mae_new = float(np.abs(hold["H"] - np.clip(ch.predict(hold[FEAT]), 1, None)).mean())
            mae_old = float(np.abs(hold["H"] - np.clip(champion.predict(hold[FEAT]), 1, None)).mean())
            promote = mae_new < 0.98 * mae_old
            if promote:
                champion = ch
            reg.append({"기준일": c, "학습 표본": len(trn), "현행 MAE": round(mae_old, 3), "도전 MAE": round(mae_new, 3), "결정": "교체" if promote else "유지"})
        return pd.DataFrame(reg)
    reg = retrain_pipeline([104, 111, 118, 125])
    print("\n[5] 자동 재학습 파이프라인 실행 기록(보류 구간 평가 후 개선 시에만 교체)")
    print(reg.to_string(index=False) if reg is not None else "실패")
    print("[6] 단계별 실행 로그:")
    for n, st, t in LOG:
        print(f"   - {n:<10} {st:<5} {t * 1000:.0f}ms")
    run_s = sum(t for _, _, t in LOG)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    piv = now.pivot_table(index="unit", columns="item", values="dos7")
    im = ax[0].imshow(piv.values, aspect="auto", cmap="RdYlGn"); ax[0].set_title("DOS heatmap (unit x item, day 100)"); plt.colorbar(im, ax=ax[0])
    ax[1].bar(["critical", "alert", "caution", "normal"], list(cnt.values()), color=["red", "orange", "gold", "tab:green"]); ax[1].set_title("Alert distribution")
    plt.tight_layout(); plt.show()

    GUIDE = ["1. 개요·KPI 정의(DOS)", "2. 입력 자료 규격·정제 규칙", "3. 실행 방법(원클릭)", "4. 경보 등급 기준", "5. 군수처장 승인 절차", "6. 설명(XAI) 보는 법", "7. 재학습 파이프라인 운영", "8. 오류 조치·연동 방법", "9. 개선 이력"]
    ok_rules = bool(rules) and all(rules.values())
    checks = [("환경 점검 4/4", env_ok == len(mods)), ("데이터 검증 5종 통과", ok_rules), ("모든 단계 OK", all(s == "OK" for _, s, _ in LOG)), ("부대 x 품목 100건 모두 분류(예측 + 이미 결핍)", len(now) + depleted == 100),
              ("자동 집행 0건", bool((q["상태"] == "군수처장 승인 대기").all())), ("재학습 파이프라인 4회 정상 실행", reg is not None and len(reg) == 4),
              ("운영 가이드 9장 완비(100%)", len(GUIDE) == 9), ("실행 시간 1분 이내", run_s < 60)]
    print("\n[7] 자체 점검")
    for n, ok in checks:
        print(f"   {'PASS' if ok else 'FAIL'}  {n}")
    npass = sum(ok for _, ok in checks)
    print("운영 가이드 목차:", " | ".join(GUIDE))
    print(f"\n[브리핑 요약] 부대 20개 x 품목 5종 DOS를 점검해 경계 이상 {len(q)}건(이미 결핍 {depleted}건 포함)을 승인 대기로 올렸다(자동 집행 없음). 보고 집계 업무는 535분 -> 34분(94% 단축 가정값).")
    print(f"재학습 {len(reg) if reg is not None else 0}회 중 교체 {int((reg['결정'] == '교체').sum()) if reg is not None else 0}회. 자체 점검 {npass}/{len(checks)} PASS. 목표(재학습 파이프라인 정상 실행, 운영 문서 100%): {'달성' if npass == len(checks) else '미달'}")


## 📋 3. 학습활동 4단계

### 활동 1: 무엇을 확인하고 계획하나요?

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 원본 목표, 기본 코드, 그리고 제공된 학습 자료를 면밀히 분석하여, "모델 교체 비교 당시 정답 H를 이미 알 수 있는 표본만 썼는가?"라는 핵심 질문에 대해 사전에 엄밀히 판단합니다.
  - 아울러 데이터 검증부터 예측, 경보, 승인 대기 분리, 가시화까지 이어지는 DOS(보급 수준) 모니터링·결핍 경보 시스템과 자동 재학습 파이프라인을 완성하는 과정에서 AI 에이전트에 위임할 분석 범위, 데이터 제약 사항, 입력 및 기간·단위·사용 가능 시점, 그리고 should_promote 함수의 입출력 계약 조건을 체계적으로 수립합니다.
- **수행 방식:**
  - 막연한 코딩이나 AI의 무분별한 답변에 의존하지 않고, 모델 교체 검증 시 미래의 정보를 사전에 참조하는 누수(Data Leakage)를 방지하기 위한 명확한 사전 규칙과 검증 계획을 문서화합니다.

#### 2. 세부 수행 절차

- **원본 목표·기본 코드·자료 분석 및 핵심 질문 검토:**
  - 데이터 검증, 예측, 경보, 승인 대기 분리, 가시화 및 자동 재학습 파이프라인을 구축하는 데 필요한 입력 데이터, 기간, 단위, 그리고 사용 가능한 시점의 제약 조건을 파악합니다.
  - 모델 교체 비교를 수행할 때 정답 H를 이미 알 수 있는 시점의 표본만을 엄격히 사용했는지 사전에 분석하고 정의합니다.
- **should_promote 입출력 계약 정의:**
  - 파라미터 mae_old , mae_new , min_improvement=0.02 등을 입력받아 교체 여부( bool )를 반환하는 핵심 함수 should_promote 의 입출력 계약 조건을 명확히 이해합니다.
  - (유한한 0 이상 오차를 검사하고 후보가 엄격히 작으면서 상대 2% 이상 개선해야 교체하며, 동률이나 현행 0은 유지하고 NaN 또는 빈 평가는 거부하는 계약 조건을 수립합니다.) AI 위임 범위 및 검증 계…

#### 완료 확인:

- 이 활동의 확인 근거 M10 계획: 데이터 검증부터 예측, 경보, 승인 대기 분리, 가시화까지 이어지는 DOS 모니터링·결핍 경보 시스템과 자동 재학습 파이프라인을 완성한다.에 필요한 입력·기간·단위·사용 가능 시점과 should_promote의입출력 계약을 설명하고 본인의사전판단·AI위임·검증계획을 기록했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 2: AI와 코드 초안을 만들어 실행하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 핵심 함수인 should_promote 를 구현·저장하고 첫 성공 실행 결과를 온전히 보존합니다.
  - 파라미터 mae_old , mae_new , min_improvement=0.02 등을 입력받아 교체 여부( bool )를 산출하며, 유한한 0 이상 오차를 검사하고 후보가 엄격히 작으면서 상대 2% 이상 개선해야 교체하고 동률이나 현행 0은 유지하며 NaN 또는 빈 평가는 거부하는 계약 조건을 준수합니다.
  - 아울러 기준 코드를 실행하고 결과를 보존한 뒤 AI 기획을 구현에 적용하고, 변경 과제(교체 문턱을 2%에서 0%로 변경하여 같은 4회 재생 실행의 선택과 오차 비교)를 수행하며, 운영 문서 9개 장 본문 작성 및 검증 실패·이미 결핍·승인 대기 경로를 점검하여 실제 코드와 출력 결과를 정확하게 확보합니다.
- **수행 방식:**
  - AI와 협업하여 자동 재학습 파이프라인 및 모델 교체 판정 로직을 코드에 반영하고, 기준 코드 실행 및 변경 과제 실험을 수행하여 신뢰성 있는 실행 결과를 확보합니다.

#### 2. 세부 수행 절차

- 핵심 함수 should_promote 구현 및 첫 성공 실행: pipeline/str01_retrain.py 내에 should_promote 함수를 구현하여 저장하고 첫 성공 실행을 보존합니다.
- mae_old , mae_new , min_improvement=0.02 인자를 통해 교체 여부( bool )를 판정하는 로직을 완성합니다.
- 유한한 0 이상 오차 검사, 후보가 엄격히 작으면서 상대 2% 이상 개선될 때만 교체, 동률 및 현행 0 유지, 그리고 NaN 이나 빈 평가 거부 계약 조건을 엄격히 반영합니다.
- 첫 성공 실행 결과, 로그, 표 및 그림 형태의 출력을 누락 없이 보존합니다.
- 변경 과제 수행 (교체 문턱 변경: 2% $\rightarrow$ 0%):…

#### 완료 확인:

- 이 활동의 확인 근거 M10 구현: pipeline/str01_retrain.py의 should_promote를저장하고계약〔mae_old,mae_new,min_improvement=.02 → 교체 여부 bool. 유한한0이상 오차를 검사하고 후보가 엄격히 작으면서 상대2% 이상 개선해야 교체한다.
- 동률·현행0은 유지하고 NaN/빈 평가를 거부한다.〕을충족하는 첫성공실행·로그·표/그림을보존했는가?
- 이후Lesson의추가변경을실제코드에반영했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 3: 원자료와 결과를 대조·수정하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - **재실행과 완료 상태**
    - 앞서 구현 및 수정된 자동 재학습 파이프라인 및 대시보드 코드를 저장하고 재실행합니다.
    - 목차 존재, 실행 성공, 운영 문서 9개 장 본문 완비, 그리고 승인 완료 상태를 명확히 구분하여 판정합니다.
  - **재학습과 모델 교체 조건**
    - 아울러 실제 기간, 분모, 조건, 수치를 바탕으로 재학습이 104/111/118/125일의 과거 재생으로 올바르게 수행되는지 검증합니다.
    - 또한 도전 학습은 기준일-14까지, 보류 평가는 기준일까지 정답이 확정된 표본만 사용하며, 2% 이상 개선 시 교체, 같은 점수면 유지, 그리고 실제 승인·집행 완료와 승인 대기 목록을 혼동하지 않음을 엄격히 확인합니다.
  - **AI 설명과 코드 검증**
    - 아울러 AI의 설명 및 코드 선택 3건 이상을 실제 군수 도메인 자료 및 실행 결과와 철저히 대조하여 보완된 답변을 도출합니다.
- **수행 방식:**
  - 단편적인 실행이나 목차 나열에 그치지 않고, 시작 문서의 FAIL 상태를 유지한 채 9개 장 본문을 모두 충실히 작성한 뒤 구조와 내용을 검토하며, 필수 검증 실패 시 예측이 올바르게 중단되는지 엄격히 교차 검증합니다.

#### 2. 세부 수행 절차

- **수정 코드 저장 및 재실행:**
  - 자동 재학습 파이프라인 및 통합 모니터링 소스 코드( pipeline/str01_retrain.py , docs/str01_operations_guide.md 등)를 안전하게 저장하고 재실행하여 에러 여부와 실행 로그를 확보합니다.
- **통합 운영 검증 및 판정 조건 준수:**
  - 자동 재학습 파이프라인이 정상 실행되고 최종 운영 문서가 100% 완비되었으며, 지속 가능하고 군수 정보체계와 연동하기 쉬운 구조인지 실제 기간·분모·조건·수치로 판정합니다.
  - 재학습 시 104/111/118/125일의 과거 재생이 정상적으로 이루어지는지 확인하고, 도전 학습 및 보류 평가 시 기준일에 따른 정답 확정 표본 제약 조건을 엄격히 적용…

#### 완료 확인:

- **판정에 필요한 근거**
  - 이 활동의 확인 근거 M10 검증: [필수] 통합 모니터링·자동 재학습 — 자동 재학습 파이프라인이 정상 실행되고 최종 문서가 100% 완비되었으며, 운영이 지속 가능하고 군수 정보체계와 연동하기 쉬운 구조인가.
  - 실제기간·분모·조건·수치로판정하고 최초/수정실행을비교했는가?
- **미션별 확인 사항**
  - **수행 조건과 판정 기준 · 학습 구간과 모델 검증**
    - 재학습은 104/111/118/125일의 과거 재생입니다.
    - 도전 학습은 기준일-14까지 정답이 확정된 표본, 보류 평가도 기준일까지 정답 확정된 표본만 씁니다.
    - 2% 이상 개선한 경우 교체하고 같은 점수면 유지합니다.
    - 승인 대기 목록은 실제 승인·집행 완료가 아닙니다.
  - **수행 조건과 판정 기준 · 원자료 대조와 검토**
    - 문서 목차 9개만으로 완료가 되지 않습니다.
    - 시작 문서 FAIL을 유지하며 9개 장 본문 작성 후 구조·내용을 검토합니다.
    - 실패 로그를 기록해도 필수 검증 실패 시 예측은 중단합니다.
- **AI 검토 기록**
  - AI 주장/선택3건을근거로검토·보완·재실행…

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:

### 활동 4: 최종 결과와 수행 상태 기록하기

**이번 활동의 과제:**

#### 1. 단계 개요

- **목표:**
  - 미션 10(전술 부대 보급 수준(DOS) 실시간 모니터링 및 결핍 경보 시스템 통합 완성)의 전 과정을 수행하며 산출된 운영 문서( docs/str01_operations_guide.md ) , 자동 재학습 파이프라인 소스 코드( pipeline/str01_retrain.py ) , 결과 데이터 파일( outputs/M10/reg.csv ) , 브리핑 보고서( reports/str01_briefing.md ) , 그리고 실습 기록 서식( forms/STR01_M10_forms.md )을 누락 없이 집대성합니다.
  - 아울러 정량적 목표 달성 여부, 미측정 영역, 문서 작성 상태 및 교관(또는 시스템) 승인 대기 상태를 명확히 구분하여 최종 판단을 기록합니다.
- **수행 방식:**
  - 단편적인 파일 제출에 치중하지 않고, 이번 미션을 통해 구축된 통합 모니터링 체계와 자동 재학습 파이프라인, 그리고 9개 장 본문이 완비된 운영 문서의 정합성을 객관적으로 검수하여 전체 PBL 실습 과정을 성공적으로 마무리할 수 있도록 최종 패키지를 정비합니다.

#### 2. 세부 수행 절차

- **핵심 산출 파일 패키지 확정 및 제출:**
  - 운영 문서인 docs/str01_operations_guide.md , 자동 재학습 소스 코드인 pipeline/str01_retrain.py , 결과 데이터 파일인 outputs/M10/reg.csv , 브리핑 보고서인 reports/str01_briefing.md , 그리고 실습 기록 서식인 forms/STR01_M10_forms.md 의 실제 작성 파일들을 최종 점검하고 제출 상태로 정리합니다.
- **코드 실행 결과 및 AI 대화 기록 연계 보존:**
  - should_promote 함수 구현 및 모델 교체 판정 결과, 그리고 분석 과정에서 AI 에이전트와 나눈 핵심 질의응답 및 코드 선택 …

#### 완료 확인:

- 이 활동의 확인 근거 M10 완료: docs/str01_operations_guide.md, pipeline/str01_retrain.py, outputs/M10/reg.csv, reports/str01_briefing.md, forms/STR01_M10_forms.md의실제작성·최종값·대화보존위치·선택이유가연결되는가?
- 미달·미측정·문서미작성·실제승인대기를정확히표시했는가?

#### 보완 방법

- 빠진 입력·파일·근거·규칙을 확인하고 관련 문서/코드를 수정해 같은 조건에서 다시 검증합니다.
- 실제 수행 기록 없이 실행·승인 완료를 만들어 넣지 않습니다.

#### 내가 기록할 것:

- 직접 확인한 입력·원본 조건:
- AI에 실제로 요청한 질문과 받은 답변의 위치:
- 실제 실행 결과/오류 메시지:
- 잘못된 설명, 수정한 코드, 수정 이유:


## 🔍 4. 평가 기준과 확인할 자료

- [정량적 평가] [필수] 통합 모니터링·자동 재학습 — 자동 재학습 파이프라인이 정상 실행되고 최종 문서가 100% 완비되었으며, 운영이 지속 가능하고 군수 정보체계와 연동하기 쉬운 구조인가. PASS 증거 제공된조건·단위·기간·분모와실제수치/건수/항목근거를연결합니다. 목표미달·0기준·미측정은그대로표시합니다. 보완 방법 기준별빠진항목과미달원인을찾아수정하고재검증합니다.
- **[정성적 평가] AI 활용: 본인 계획·실제 대화·코드 변경·검토3건·보완·재실행·전후 비교·최종 선택.**
  - **PASS 증거**
    - **수행 조건과 판정 기준 · 학습 구간과 모델 검증**
      - 재학습은 104/111/118/125일의 과거 재생입니다.
      - 도전 학습은 기준일-14까지 정답이 확정된 표본, 보류 평가도 기준일까지 정답 확정된 표본만 씁니다.
      - 2% 이상 개선한 경우 교체하고 같은 점수면 유지합니다.
      - 승인 대기 목록은 실제 승인·집행 완료가 아닙니다.
    - **수행 조건과 판정 기준 · 원자료 대조와 검토**
      - 문서 목차 9개만으로 완료가 되지 않습니다.
      - 시작 문서 FAIL을 유지하며 9개 장 본문 작성 후 구조·내용을 검토합니다.
      - 실패 로그를 기록해도 필수 검증 실패 시 예측은 중단합니다.
    - **보완 방법**
      - 보완 방법 근거없는원인·규정·완료추정은수정하거나확인필요로보류합니다.
- **[진행 추적 및 완료 기준] 정성·검증:**
  - **수행 조건과 판정 기준 · 학습 구간과 모델 검증**
    - 재학습은 104/111/118/125일의 과거 재생입니다.
    - 도전 학습은 기준일-14까지 정답이 확정된 표본, 보류 평가도 기준일까지 정답 확정된 표본만 씁니다.
    - 2% 이상 개선한 경우 교체하고 같은 점수면 유지합니다.
    - 승인 대기 목록은 실제 승인·집행 완료가 아닙니다.
  - **수행 조건과 판정 기준 · 원자료 대조와 검토**
    - 문서 목차 9개만으로 완료가 되지 않습니다.
    - 시작 문서 FAIL을 유지하며 9개 장 본문 작성 후 구조·내용을 검토합니다.
    - 실패 로그를 기록해도 필수 검증 실패 시 예측은 중단합니다.
  - **제출 파일 및 완료 기록**
    - 파일: docs/str01_operations_guide.md, pipeline/str01_retrain.py, outputs/M10/reg.csv, reports/str01_briefing.md, forms/STR01_M10_forms.md 실제AI요청·답변·근거대조3건이상·보완·재확인·본인최종선택과작성/실행기록이연결됩니다.
    - 최종ZIP생성은실제학습완료·교관승인 인증이아닙니다.
  - **보완 방법**
    - 보완 방법 미작성·미실행·검토누락을보완하고교관검토상태를별도로표시합니다.
**실제 출력 없이 달성으로 표시하지 않습니다.**

In [ ]:
# @title ⑤ 개별 산출물·학습기록 통합 입력, 저장 및 ZIP 다운로드 (실행 버튼을 누르세요) { display-mode: "form" }
# 입력 폼이 나타나면 소스코드 대신 버튼을 이용하세요. Colab/Jupyter에서 ipywidgets 필요.
from pathlib import Path

if not (globals().get('STR01_PROJECT_READY', False) and globals().get('STR01_PROJECT_DIR') and Path(STR01_PROJECT_DIR).is_dir()):
    print('먼저 ① 실습자료 자동 준비 및 환경 설정 셀을 실행한 뒤 이 셀을 다시 실행하세요. ZIP 업로드는 필요 없습니다.')
else:
    def _load_notebook_widgets():
        from importlib import import_module, invalidate_caches
        import subprocess
        import sys

        try:
            return import_module('ipywidgets')
        except ImportError:
            print('이 노트북 커널에 ipywidgets가 없어 설치합니다.')

        try:
            subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                 '--retries', '0', '--timeout', '10', 'ipywidgets'],
                check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=60,
            )
            invalidate_caches()
            return import_module('ipywidgets')
        except (subprocess.SubprocessError, OSError, ImportError):
            print('이 커널에서 ipywidgets를 설치하거나 불러오지 못했습니다.')
            print('설치 가능한 환경에서는 새 셀에서 %pip install ipywidgets 를 실행한 뒤 이 셀을 다시 실행하세요.')
            print('설치할 수 없는 환경에서는 제공된 Markdown 작성 양식을 사용하세요.')
            return None

    widgets = _load_notebook_widgets()
    if widgets is not None:
        import json, csv, io, ast, zipfile, datetime, pathlib
        from pathlib import Path
        import os
        from urllib.parse import quote
        from IPython.display import HTML, display, Markdown, clear_output

        MISSION_CODE = 10
        DELIVERABLE_PATHS = ['docs/str01_operations_guide.md', 'pipeline/str01_retrain.py', 'outputs/M10/reg.csv', 'reports/str01_briefing.md', 'forms/STR01_M10_forms.md']
        PROJECT_NAME = 'STR01'
        WORK_ROOT = Path(STR01_PROJECT_DIR).resolve()

        RECORD_DIR = WORK_ROOT / f'STR01_M{MISSION_CODE:02d}_학습기록'
        RECORD_DIR.mkdir(parents=True, exist_ok=True)
        ZIP_FILENAME = f'STR01_m{MISSION_CODE:02d}_review.zip'

        # 반드시 CMS 지정 경로만 사용하며 상대 경로의 작업공간 이탈을 방지합니다.
        def safe_path(relative):
            target = (WORK_ROOT / relative).resolve()
            if not target.is_relative_to(WORK_ROOT):
                raise ValueError('작업공간 밖의 경로는 사용할 수 없습니다.')
            return target

        def valid_text(path, content):
            suffix = Path(path).suffix.lower()
            if suffix == '.json':
                json.loads(content)
            elif suffix == '.ipynb':
                doc = json.loads(content)
                if not isinstance(doc, dict) or not isinstance(doc.get('cells'), list) or 'nbformat' not in doc:
                    raise ValueError('노트북 JSON에 cells 또는 nbformat 정보가 없습니다.')
            elif suffix == '.py':
                ast.parse(content)
            elif suffix == '.csv':
                if not content.strip():
                    raise ValueError('CSV 데이터가 비어 있습니다.')
                rows = list(csv.reader(io.StringIO(content)))
                if not rows or not rows[0] or not any(x.strip() for x in rows[0]):
                    raise ValueError('CSV 첫 행에 열 이름을 입력하세요.')
                if any(len(row) != len(rows[0]) for row in rows if row):
                    raise ValueError('CSV 열 개수가 행마다 다릅니다.')

        # 산출물 입력 화면 — 파일별 별도 탭
        editors = {}
        file_tabs = []
        file_buttons = {}
        for relative in DELIVERABLE_PATHS:
            suffix = Path(relative).suffix.lower()
            hint = {
                '.md': '보고서·양식의 제목, 분석 결과, 근거와 수정 이유를 Markdown으로 작성합니다.',
                '.csv': '첫 줄에 열 이름, 다음 줄부터 데이터 행을 쉼표로 구분하여 입력합니다.',
                '.json': '키와 값이 있는 유효한 JSON 형식으로 실제 결과를 입력합니다.',
                '.py': '직접 구현하고 확인한 Python 소스코드를 입력합니다.',
                '.ipynb': '노트북 JSON 원문을 입력하거나 기존 파일에서 불러옵니다. JSON 노트북 형식을 유지하세요.'
            }.get(suffix, '실제 작성한 내용을 입력합니다.')
            editor = widgets.Textarea(value='', placeholder='여기에 내용을 작성하세요. 입력하지 않은 파일은 저장하지 않습니다.',
                                      layout=widgets.Layout(width='100%', height='250px'))
            status = widgets.HTML('<span style="color:#777">아직 저장하지 않았습니다.</span>')
            load_btn = widgets.Button(description='기존 파일 불러오기', icon='folder-open', button_style='')
            save_btn = widgets.Button(description='이 파일 저장', icon='save', button_style='success')
            editors[relative] = {'editor': editor, 'status': status, 'loaded': False, 'saved': False, 'dirty': False, 'saved_value': ''}
            def on_edit(change, key=relative):
                state = editors[key]
                if state['loaded']:
                    state['dirty'] = True
                    state['status'].value = '<span style="color:#a65">● 수정됨 — 저장 버튼을 누르세요.</span>'
            editor.observe(on_edit, names='value')
            def load_clicked(button, key=relative):
                state = editors[key]
                target = safe_path(key)
                if not target.is_file():
                    state['status'].value = '<span style="color:#a22">파일이 아직 없습니다. 직접 입력 후 저장하세요.</span>'
                    return
                if state['dirty']:
                    state['status'].value = '<span style="color:#a22">미저장 내용이 있습니다. 불러오기 전에 저장하세요.</span>'
                    return
                try:
                    state['loaded'] = False
                    state['saved_value'] = target.read_text(encoding='utf-8')
                    state['editor'].value = state['saved_value']
                    state['loaded'] = True
                    state['dirty'] = False
                    state['saved'] = True
                    state['status'].value = '<span style="color:#286">파일을 불러왔습니다. 변경하면 다시 저장하세요.</span>'
                except Exception as exc:
                    state['status'].value = '<span style="color:#a22">불러오기 오류: '+str(exc)+'</span>'
            def save_clicked(button, key=relative):
                save_one(key)
            load_btn.on_click(load_clicked)
            save_btn.on_click(save_clicked)
            file_buttons[relative] = {'load':load_btn,'save':save_btn}
            file_tabs.append(widgets.VBox([
                widgets.HTML('<b>저장 경로:</b> <code>'+relative+'</code><br><span>'+hint+'</span><br><span style="color:#a55">기존 파일이 있다면 저장 시 덮어씁니다. 실습 완료·승인과는 다릅니다.</span>'),
                editor, widgets.HBox([load_btn, save_btn]), status
            ]))

        def save_one(relative):
            state = editors[relative]
            value = state['editor'].value
            if not value.strip():
                state['status'].value = '<span style="color:#a22">비어 있어 저장하지 않았습니다.</span>'
                return False
            try:
                valid_text(relative, value)
                target = safe_path(relative)
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_text(value, encoding='utf-8')
                state['saved_value'] = value
                state['loaded'] = True
                state['dirty'] = False
                state['saved'] = True
                state['status'].value = '<span style="color:green">✓ 저장 완료: '+relative+'</span>'
                return True
            except Exception as exc:
                state['status'].value = '<span style="color:#a22">저장 실패: '+str(exc)+'</span>'
                return False

        # 학습기록 및 AI 원문, 자료 대조, 검토 3건 이상
        record_specs = [
         ('plan','학습 계획·수행 기록','이번 미션에서 직접 수행한 내용, 실행 조건 및 최초 결과를 작성하세요.'),
         ('ai_request','실제 AI 질문 원문','실제로 입력한 질문 원문만 작성하세요. 사용하지 않았다면 비워 둡니다.'),
         ('ai_response','실제 받은 AI 답변 원문','실제로 받은 답변 전체를 붙여 넣으세요. 임의로 생성한 대화는 기록하지 않습니다.'),
         ('evidence','실제 출력·자료 대조 근거','실제 출력값, 원자료 위치, 전후 비교 및 계산 근거를 적으세요.'),
         ('revision','검토 및 수정 내용 종합','검토 3건의 발견사항과 코드를 수정·재실행한 기록을 종합하세요.'),
         ('decision','최종 판단과 미달·미측정·미실시 사유','달성 여부와 남은 부분, 근거 및 추가 확인사항을 적으세요.'),
        ]
        record_widgets = {}
        record_items = []
        for key, title, hint in record_specs:
            field = widgets.Textarea(placeholder=hint, layout=widgets.Layout(width='100%',height='115px'))
            record_widgets[key] = field
            record_items.append(widgets.VBox([widgets.HTML('<b>'+title+'</b><br><small>'+hint+'</small>'),field]))

        reviews = []
        for i in range(3):
            claim = widgets.Textarea(placeholder=f'{i+1}번째 AI 주장·코드 선택 또는 점검 대상',layout=widgets.Layout(width='100%',height='65px'))
            ground = widgets.Textarea(placeholder='원자료·실행 출력의 위치 및 비교한 근거',layout=widgets.Layout(width='100%',height='65px'))
            action = widgets.Textarea(placeholder='맞음·수정·보류, 수정 내용 및 재확인 결과',layout=widgets.Layout(width='100%',height='65px'))
            reviews.append((claim,ground,action))
            record_items.append(widgets.VBox([widgets.HTML(f'<b>검토 {i+1} (최소 3건 중 {i+1}번째)</b>'),claim,ground,action]))

        status_check = widgets.SelectMultiple(options=['달성','미달','미측정','미실시','추가 확인 필요'],
                                              value=(),description='수행 상태',layout=widgets.Layout(width='70%',height='115px'))
        approval = widgets.Dropdown(options=['미확인 / 교관 승인 대기','실제 교관 검토 완료 (기록 확인 필요)'],
                                    value='미확인 / 교관 승인 대기', description='교관 상태',layout=widgets.Layout(width='85%'))
        record_notice = widgets.HTML('<span style="color:#666">입력 후 [학습기록 저장]을 누르세요. 미작성·미측정은 그대로 표시합니다.</span>')
        record_dirty = {'value': False}
        def _mark_record_dirty(change):
            record_dirty['value'] = True
            record_notice.value = '<span style="color:#a65">● 학습기록에 저장하지 않은 변경이 있습니다.</span>'
        for field in list(record_widgets.values()) + [v for group in reviews for v in group] + [status_check,approval]:
            field.observe(_mark_record_dirty,names='value')

        # 앞서 작성한 학습기록을 자동으로 불러와 이어서 수정할 수 있습니다.
        previous_audit = RECORD_DIR/'learner_audit.json'
        if previous_audit.is_file():
            try:
                previous = json.loads(previous_audit.read_text(encoding='utf-8'))
                for key, field in record_widgets.items():
                    field.value = str(previous.get(key,'') or '')
                for group, past in zip(reviews, previous.get('reviews',[])):
                    for widget, key in zip(group,('claim','evidence','revision_and_recheck')):
                        widget.value = str(past.get(key,'') or '')
                status_check.value = tuple(x for x in previous.get('status',[]) if x in status_check.options)
                if previous.get('instructor_approval') in approval.options:
                    approval.value = previous['instructor_approval']
                record_dirty['value'] = False
                record_notice.value = '<span style="color:#286">이전 학습기록을 불러왔습니다. 수정 후 다시 저장하세요.</span>'
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">기존 학습기록 읽기 오류: '+str(exc)+'</span>'

        def audit_object():
            return {
                'project': PROJECT_NAME, 'mission': MISSION_CODE,
                'plan':record_widgets['plan'].value, 'ai_request':record_widgets['ai_request'].value,
                'ai_response':record_widgets['ai_response'].value, 'evidence':record_widgets['evidence'].value,
                'revision':record_widgets['revision'].value, 'decision':record_widgets['decision'].value,
                'reviews':[{'claim':a.value,'evidence':b.value,'revision_and_recheck':c.value} for a,b,c in reviews],
                'status':list(status_check.value),'instructor_approval':approval.value,
                'saved_at':datetime.datetime.now().isoformat(timespec='seconds')
            }

        def save_audit(button=None):
            try:
                record = audit_object()
                (RECORD_DIR/'learner_audit.json').write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf-8')
                (RECORD_DIR/'AI_대화원문.txt').write_text('실제 AI 질문 원문\n'+record['ai_request']+'\n\n실제 AI 답변 원문\n'+record['ai_response'],encoding='utf-8')
                sections = [f'# STR01 미션 {MISSION_CODE:02d} 학습기록']
                for name,key in [('학습 계획·수행 기록','plan'),('실제 출력·자료 대조 근거','evidence'),('검토 및 수정 내용','revision'),('최종 판단','decision')]:
                    sections.append('## '+name+'\n'+(record[key] or '(미작성)'))
                for i,item in enumerate(record['reviews'],1):
                    sections.append(f"## 검토 {i}\n- 검토 대상: {item['claim'] or '(미작성)'}\n- 대조 근거: {item['evidence'] or '(미작성)'}\n- 수정 및 재확인: {item['revision_and_recheck'] or '(미작성)'}")
                sections.append('## 상태\n'+(', '.join(record['status']) or '미선택')+'\n\n교관 상태: '+record['instructor_approval'])
                (RECORD_DIR/'학습기록.md').write_text('\n\n'.join(sections)+'\n',encoding='utf-8')
                record_dirty['value'] = False
                completed = sum(all(v.strip() for v in [a.value,b.value,c.value]) for a,b,c in reviews)
                record_notice.value = f'<span style="color:green">✓ 학습기록 저장 완료. 근거가 모두 입력된 검토: {completed}/3건. 저장은 PASS나 교관 승인과 다릅니다.</span>'
                return True
            except Exception as exc:
                record_notice.value = '<span style="color:#a22">학습기록 저장 실패: '+str(exc)+'</span>'
                return False

        save_audit_btn = widgets.Button(description='학습기록 저장',icon='save',button_style='success')
        save_audit_btn.on_click(save_audit)
        record_tab = widgets.VBox(record_items+[widgets.HTML('<b>최종 수행 상태</b> — 해당되는 항목을 선택하세요. 선택만으로 자동 평가되지 않습니다.'),status_check,approval,save_audit_btn,record_notice])

        tabs = widgets.Tab(children=file_tabs+[record_tab])
        for i, path in enumerate(DELIVERABLE_PATHS):
            tabs.set_title(i, f'산출물 {i+1}')
        tabs.set_title(len(file_tabs),'학습기록')

        summary_output = widgets.Output()
        all_btn = widgets.Button(description='작성한 산출물 전체 저장', icon='save',button_style='info')
        zip_btn = widgets.Button(description='저장 파일 ZIP 생성',icon='archive',button_style='warning')
        dl_btn = widgets.Button(description='ZIP 다운로드',icon='download',button_style='')

        def save_all(button):
            ok = 0
            for rel in DELIVERABLE_PATHS:
                if editors[rel]['editor'].value.strip() and save_one(rel):
                    ok += 1
            with summary_output:
                summary_output.clear_output()
                print(f'산출물 {ok}개 저장 완료. 파일별 상태를 확인하세요. 학습기록은 별도 [학습기록 저장] 버튼을 누르세요.')

        def build_zip(button):
            with summary_output:
                summary_output.clear_output()
                if record_dirty['value']:
                    print('❌ 학습기록에 미저장 변경 내용이 있습니다. [학습기록 저장]을 먼저 누르세요.')
                    return
                dirty = [p for p,s in editors.items() if s['dirty']]
                if dirty:
                    print('❌ 미저장 변경 내용이 있습니다. 아래 파일들을 먼저 저장하세요:')
                    for p in dirty: print(' -',p)
                    return
                # 기존에 제공된 파일은 ZIP에 임의로 포함하지 않고 현재 화면에서 불러오거나 저장한 파일만 수집합니다.
                saved = [p for p,s in editors.items() if s['saved'] and safe_path(p).is_file()]
                audit_files = [p for p in RECORD_DIR.rglob('*') if p.is_file()]
                if not saved and not audit_files:
                    print('❌ 저장된 결과물이 없습니다. 산출물 또는 학습기록을 저장하세요.')
                    return
                dest = WORK_ROOT / ZIP_FILENAME
                with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as archive:
                    for rel in saved: archive.write(safe_path(rel), arcname=rel)
                    for f in audit_files: archive.write(f, arcname=str(f.relative_to(WORK_ROOT)))
                print('✅ ZIP 생성:', dest)
                print('산출물:',len(saved),'개 / 학습기록:',len(audit_files),'개')
                print('※ ZIP 생성은 학습 완료 또는 교관 승인을 의미하지 않습니다.')

        def download_zip(button):
            with summary_output:
                summary_output.clear_output()
                target = WORK_ROOT / ZIP_FILENAME
                if not target.is_file():
                    print('먼저 [저장 파일 ZIP 생성]을 눌러 주세요.')
                    return
                try:
                    from google.colab import files as colab_files
                    colab_files.download(str(target))
                    print('다운로드를 요청했습니다.')
                except ImportError:
                    start_dir = globals().get('notebook_directory', WORK_ROOT.parent)
                    href = quote(Path(os.path.relpath(target, start_dir)).as_posix())
                    display(HTML(f'<a href="{href}" download>ZIP 다운로드: {target.name}</a>'))

        all_btn.on_click(save_all)
        zip_btn.on_click(build_zip)
        dl_btn.on_click(download_zip)

        print(f'✅ STR01 미션 {MISSION_CODE:02d} 산출물·학습기록 입력 화면')
        display(Markdown('**사용 순서:** ① 각 산출물 탭에서 작성하거나 기존 파일 불러오기 → ② 파일별 저장 → ③ 학습기록 탭에서 실제 AI 원문·근거·검토 3건·최종 상태 입력 및 저장 → ④ ZIP 생성 → ⑤ 다운로드.'))
        display(Markdown('**주의:** 원본 실습의 TODO는 직접 실행해야 합니다. 입력창은 실제 수행 내용을 기록하기 위한 기능이며, 미작성 결과를 자동으로 완성하지 않습니다.'))
        display(tabs,widgets.HBox([all_btn,zip_btn,dl_btn]),summary_output)
